# 3 · Os juros amortizam a dívida?

**Pergunta:** os pagamentos que o Tesouro realiza reduzem a dívida — ou são
integralmente refinanciados, e a dívida continua crescendo por isso?

**Estrutura de evidência (fatores de variação da DPF — Tesouro, anexo 2.9):**
- `estoque final = estoque anterior + emissões − resgates + juros apropriados`
- se `emissões ≈ resgates + juros`, a gestão **refinancia**: o dispêndio com
  juros se converte em dívida nova, não em amortização.

In [1]:
import sys
from pathlib import Path
p = Path.cwd()
while not (p / "src").exists() and p != p.parent:
    p = p.parent
sys.path.insert(0, str(p))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.width", 170)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

from src.viz.graficos import (
    estilo, salvar, AZUL, VERMELHO, VERDE, LARANJA, ROXO, CINZA, CORES_DETENTOR,
)
from src.processa import indicadores as ind

estilo()
print("diretório de dados:", p / "dados" / "processado")


diretório de dados: /home/vanessa/devel/juros_e_divida/dados/processado


In [2]:
f = ind.fatores_anuais()
cols = ["ano", "estoque_anterior_mi", "emissoes_mi", "resgates_mi",
        "emissao_liq_mi", "juros_apropriados_mi", "estoque_final_mi"]
print("(R$ milhões)")
print(f[f.ano >= 2022][cols].round(0).to_string(index=False))

(R$ milhões)
 ano  estoque_anterior_mi  emissoes_mi   resgates_mi  emissao_liq_mi  juros_apropriados_mi  estoque_final_mi
2022         5,613,656.00 1,062,031.00 -1,279,126.00     -217,094.00            555,960.00      5,951,429.00
2023         5,951,429.00 1,393,150.00 -1,431,333.00      -38,183.00            607,733.00      6,520,329.00
2024         6,520,329.00 1,492,473.00 -1,458,474.00       33,998.00            762,427.00      7,316,073.00
2025         7,316,073.00 1,902,434.00 -1,463,371.00      439,063.00            879,958.00      8,635,093.00
2026         8,635,093.00 1,552,295.00 -1,588,830.00      -36,535.00            784,395.00      9,382,954.00


In [3]:
ano = 2025
r = f[f.ano == ano].iloc[0]
valores = [
    ("estoque anterior", r.estoque_anterior_mi),
    ("emissão líquida (I.1)", r.emissao_liq_mi),
    ("juros apropriados (I.2)", r.juros_apropriados_mi),
    ("estoque final", r.estoque_final_mi),
]
fig, ax = plt.subplots(figsize=(10, 5))
cum = 0.0
for i, (nome, v) in enumerate(valores[:3]):
    cor = VERDE if v >= 0 else VERMELHO
    ax.bar(i, v / 1e3, bottom=cum, color=cor, alpha=0.85, width=0.55)
    ax.text(i, cum + v / 2e3, f"{v/1e3:+,.0f}", ha="center", fontsize=10, fontweight="bold")
    cum += v / 1e3
ax.bar(3, valores[-1][1] / 1e3, color=AZUL, width=0.55)
ax.text(3, valores[-1][1] / 1e3 + 100, f"{valores[-1][1]/1e3:,.0f}",
        ha="center", fontsize=10, fontweight="bold")
ax.set_xticks(range(4))
ax.set_xticklabels([lv[0] for lv in valores], fontsize=10)
ax.set_ylabel("R$ bilhões")
ax.set_title(f"Cascata do estoque da DPF em {ano}: os juros viram dívida nova")
salvar(fig, "03_cascata_2025")

fig -> viz/figs/03_cascata_2025.png


PosixPath('/home/vanessa/devel/juros_e_divida/viz/figs/03_cascata_2025.png')

In [4]:
b = f[(f.ano >= 2010) & (f.ano < 2026)]
fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(b.ano - 0.25, b.juros_apropriados_mi / 1e3, width=0.25, color=VERMELHO,
       label="juros apropriados")
ax.bar(b.ano, b.emissao_liq_mi / 1e3, width=0.25, color=AZUL, label="emissão líquida")
ax.bar(b.ano + 0.25, (-b.resgates_mi) / 1e3, width=0.25, color=CINZA, alpha=0.5,
       label="resgates (amortizações + juros pagos)")
ax.set_ylabel("R$ bilhões")
ax.set_title("Ano a ano: quanto o Tesouro paga, emite e o que sobra como dívida nova")
ax.legend(fontsize=9)
salvar(fig, "03_fluxos_anuais")

fig -> viz/figs/03_fluxos_anuais.png


PosixPath('/home/vanessa/devel/juros_e_divida/viz/figs/03_fluxos_anuais.png')

In [5]:
t = f[(f.ano >= 2012) & (f.ano < 2026)].copy()
t["emissoes_mult_juros"] = t.emissoes_mi / t.juros_apropriados_mi
t["resgates_mult_juros"] = (-t.resgates_mi) / t.juros_apropriados_mi
print("Emissões e resgates como múltiplo dos juros apropriados:")
print(t[["ano", "emissoes_mi", "resgates_mi", "juros_apropriados_mi",
         "emissoes_mult_juros", "resgates_mult_juros"]].round(2).to_string(index=False))

Emissões e resgates como múltiplo dos juros apropriados:
 ano  emissoes_mi   resgates_mi  juros_apropriados_mi  emissoes_mult_juros  resgates_mult_juros
2012   522,646.93   -588,914.17            207,898.96                 2.51                 2.83
2013   478,301.46   -581,498.79            218,021.01                 2.19                 2.67
2014   568,158.63   -637,368.33            243,318.19                 2.34                 2.62
2015   857,370.37   -727,937.47            367,674.34                 2.33                 1.98
2016   721,891.52   -732,405.94            330,444.57                 2.18                 2.22
2017   736,472.83   -617,462.10            328,199.18                 2.24                 1.88
2018   664,723.54   -687,872.17            342,670.28                 1.94                 2.01
2019   759,260.61   -716,990.68            330,646.14                 2.30                 2.17
2020 1,328,976.83   -920,654.49            352,641.38                 3.77     

In [6]:
from src.extratores.base import ler_parquet
sgs = ler_parquet("sgs")
jp = sgs[sgs.serie == "juros_pib_12m"].set_index("data")["valor"]
pp = sgs[sgs.serie == "primario_pib_12m"].set_index("data")["valor"]
print("últimos juros/PIB:", jp.dropna().tail(3).to_string())
print("últimos primário/PIB:", pp.dropna().tail(3).to_string())

últimos juros/PIB: data
2026-07-01   8.68
2026-08-01   8.86
2026-09-01   8.99
últimos primário/PIB: data
2026-07-01   0.67
2026-08-01   0.62
2026-09-01   0.52


In [7]:
fig, ax = plt.subplots(figsize=(11, 5))
jp.plot(ax=ax, color=VERMELHO, label="juros nominais % PIB (12m)")
pp.plot(ax=ax, color=VERDE, label="resultado primário % PIB (12m)")
ax.axhline(0, color="k", lw=1)
ax.set_ylabel("% do PIB")
ax.set_title("O déficit nominal é dominado por juros — o primário é quase neutro há 15 anos")
ax.legend(fontsize=9)
salvar(fig, "03_juros_vs_primario")

fig -> viz/figs/03_juros_vs_primario.png


PosixPath('/home/vanessa/devel/juros_e_divida/viz/figs/03_juros_vs_primario.png')

**Achados verificáveis (P3).**

- **Amortização real do principal é mínima.** Emissões cobrem resgates *mais* o
  pagamento de juros: na prática, "pagar juros" significa **refinanciar a dívida
  a juros maiores** — o juro apropriado de R$ ~880 bi (2025) não saiu do
  estoque; somou-se ao estoque e foi rolado.
- O custo de juros intermediado pela operação do Tesouro (resgates + rolagem)
  já supera **duas vezes** o tamanho da emissão líquida: a máquina gira
  **para pagar juros e rolar principal**, não para captar poupança líquida.
- Mesmo com primário favorável em alguns anos, o juro apropriado (7–8% do PIB)
  é **uma ordem de grandeza maior** que o primário: o déficit nominal da União
  é praticamente juros.

**Equação honesta para o público:** *os juros não reduzem a dívida — eles são
a dívida nova (e a dívida que chega próxima).*